In [ ]:
import pandas as pd

df = pd.read_csv("smart_infrastructure_risk_dataset.csv")

In [ ]:
df[['complaint_id','severity','complaint_frequency','damage_size',
    'previous_accidents','repair_delay_days','risk_score']].head(10)

,complaint_id,severity,complaint_frequency,damage_size,previous_accidents,repair_delay_days,risk_score
0,1,high,40,7.17,9,126,73.06
1,2,medium,30,5.36,8,55,48.87
2,3,medium,30,9.17,15,162,81.78
3,4,high,41,8.16,13,100,79.38
4,5,low,11,1.99,0,17,16.40
5,6,high,15,6.99,5,92,54.50
6,7,low,12,1.99,3,35,19.66
7,8,high,25,8.63,14,149,82.14
8,9,medium,16,4.07,3,66,34.12
9,10,medium,32,3.63,6,31,42.79


In [ ]:
dummies = pd.get_dummies(df['severity'])
dummies.head(10)

,high,low,medium
0,True,False,False
1,False,False,True
2,False,False,True
3,True,False,False
4,False,True,False
5,True,False,False
6,False,True,False
7,True,False,False
8,False,False,True
9,False,False,True


In [ ]:
merged = pd.concat([df, dummies], axis='columns')
merged.drop('severity', axis='columns', inplace=True)
merged[['complaint_id','low','medium','high',
        'complaint_frequency','damage_size','risk_score']].head(10)

,complaint_id,low,medium,high,complaint_frequency,damage_size,risk_score
0,1,False,False,True,40,7.17,73.06
1,2,False,True,False,30,5.36,48.87
2,3,False,True,False,30,9.17,81.78
3,4,False,False,True,41,8.16,79.38
4,5,True,False,False,11,1.99,16.40
5,6,False,False,True,15,6.99,54.50
6,7,True,False,False,12,1.99,19.66
7,8,False,False,True,25,8.63,82.14
8,9,False,True,False,16,4.07,34.12
9,10,False,True,False,32,3.63,42.79


In [ ]:
final = merged.drop('low', axis='columns')

final[['complaint_id','medium','high',
       'complaint_frequency','damage_size','risk_score']].head(10)

,complaint_id,medium,high,complaint_frequency,damage_size,risk_score
0,1,False,True,40,7.17,73.06
1,2,True,False,30,5.36,48.87
2,3,True,False,30,9.17,81.78
3,4,False,True,41,8.16,79.38
4,5,False,False,11,1.99,16.40
5,6,False,True,15,6.99,54.50
6,7,False,False,12,1.99,19.66
7,8,False,True,25,8.63,82.14
8,9,True,False,16,4.07,34.12
9,10,True,False,32,3.63,42.79


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_le = df[['severity','complaint_frequency','damage_size',
            'previous_accidents','repair_delay_days','risk_score']].copy()
df_le['severity_encoded'] = le.fit_transform(df_le['severity'])


df_le[['severity','severity_encoded','complaint_frequency','damage_size','risk_score']].head(10)

,severity,severity_encoded,complaint_frequency,damage_size,risk_score
0,high,0,40,7.17,73.06
1,medium,2,30,5.36,48.87
2,medium,2,30,9.17,81.78
3,high,0,41,8.16,79.38
4,low,1,11,1.99,16.40
5,high,0,15,6.99,54.50
6,low,1,12,1.99,19.66
7,high,0,25,8.63,82.14
8,medium,2,16,4.07,34.12
9,medium,2,32,3.63,42.79


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np


df_model = df[['severity','complaint_frequency','damage_size',
               'previous_accidents','repair_delay_days']].copy()
le2 = LabelEncoder()
df_model['severity'] = le2.fit_transform(df_model['severity'])

X = df_model.values
y = df['risk_score'].values


print(X[:5])

[[  0.    40.     7.17   9.   126.  ]
 [  2.    30.     5.36   8.    55.  ]
 [  2.    30.     9.17  15.   162.  ]
 [  0.    41.     8.16  13.   100.  ]
 [  1.    11.     1.99   0.    17.  ]]


In [ ]:
ct = ColumnTransformer([('severity', OneHotEncoder(), [0])], remainder='passthrough')
X_encoded = ct.fit_transform(X)


print(X_encoded[:5])

[[  1.     0.     0.    40.     7.17   9.   126.  ]
 [  0.     0.     1.    30.     5.36   8.    55.  ]
 [  0.     0.     1.    30.     9.17  15.   162.  ]
 [  1.     0.     0.    41.     8.16  13.   100.  ]
 [  0.     1.     0.    11.     1.99   0.    17.  ]]


In [ ]:
X_encoded = X_encoded[:, 1:]
print(X_encoded[:5])

[[  0.     0.    40.     7.17   9.   126.  ]
 [  0.     1.    30.     5.36   8.    55.  ]
 [  0.     1.    30.     9.17  15.   162.  ]
 [  0.     0.    41.     8.16  13.   100.  ]
 [  1.     0.    11.     1.99   0.    17.  ]]


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_encoded, y)

LinearRegression()

High severity


In [ ]:
pred1 = model.predict([[0, 0, 45, 9.5, 12, 160]])
print(f"{pred1[0]:.2f}")

90.32


Medium severity



In [ ]:
pred2 = model.predict([[0, 1, 25, 5.0, 6, 90]])
print(f"{pred2[0]:.2f}")

48.62


Low severity

In [ ]:
pred3 = model.predict([[1, 0, 5, 1.5, 1, 20]])

In [ ]:
print(f"{pred3[0]:.2f}")

11.98
